# CDR Cleaning Practice Notebook

This notebook is a scaffold for practicing data cleaning on the CDR CSV in `cleaned data/`.

Use it to track each edit, validate each transformation, and keep a clear record of what changed in each pass.

## Session Guide

1. Load the canonical CSV.
2. Audit shape, dtypes, and missing values.
3. Standardize column names.
4. Clean text fields and inspect encoding issues.
5. Build ontology seed tables for chemicals and companies.
6. Convert types and validate results.
7. Export a versioned working copy when you are ready.

Keep each step small so you can explain what changed later.

In [10]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display


def log(message: str) -> None:
    stamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{stamp}] {message}")


session_log = []


def record_change(message: str) -> None:
    entry = f"{datetime.now():%Y-%m-%d %H:%M:%S} | {message}"
    session_log.append(entry)
    print(entry)


record_change('Notebook scaffold initialized.')
log('Imports loaded successfully.')

2026-06-20 05:57:58 | Notebook scaffold initialized.
[2026-06-20 05:57:58] Imports loaded successfully.


In [11]:
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / 'cleaned data' / '2024 CDR Consumer and Commercial Use Information_clean.csv'
OUTPUT_DIR = PROJECT_ROOT / 'cleaned data' / 'exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log(f'Project root: {PROJECT_ROOT}')
log(f'Input file: {DATA_PATH}')
log(f'Export directory: {OUTPUT_DIR}')

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing expected source file: {DATA_PATH}')

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
df = raw_df.copy()

log(f'Loaded dataframe with {df.shape[0]:,} rows and {df.shape[1]:,} columns.')
record_change(f'Loaded {DATA_PATH.name} with shape {df.shape[0]:,} x {df.shape[1]:,}.')
display(df.head(3))

[2026-06-20 05:57:58] Project root: c:\Users\Willaim\Desktop\src\Exce Workl Project
[2026-06-20 05:57:58] Input file: c:\Users\Willaim\Desktop\src\Exce Workl Project\cleaned data\2024 CDR Consumer and Commercial Use Information_clean.csv
[2026-06-20 05:57:58] Export directory: c:\Users\Willaim\Desktop\src\Exce Workl Project\cleaned data\exports
[2026-06-20 05:57:59] Loaded dataframe with 64,023 rows and 85 columns.
2026-06-20 05:57:59 | Loaded 2024 CDR Consumer and Commercial Use Information_clean.csv with shape 64,023 x 85.


,CHEMICAL NAME,CHEMICAL ID,CHEMICAL ID W/O DASHES,CHEMICAL ID TYPE,STANDARDIZED PARENT COMPANY NAME,FOREIGN PARENT COMPANY NAME,FOREIGN PC ADDRESS LINE1,FOREIGN PC ADDRESS LINE2,FOREIGN PC CITY,FOREIGN PC COUNTY / PARISH,...,JOINT FC CODE,JOINT FUNCTION CATEGORY,JOINT FUNCT CAT OTHER DESC,CONS AND/OR COMM USE,USED IN PROD FOR CHILDREN,C / C PV PCT,C / C MAX CONC CODE,CONS / COMM MAXIMUM CONCENTRATION,COMM WORKERS CODE,COMMERCIAL WORKERS REASONABLY LIKELY EXPOSED
0,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,4193559,CASRN,TEH FONG MIN INTERNATIONAL,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Both,NKRA,10%,M1,< 1%,W1,< 10
1,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,41098560,CASRN,TEH FONG MIN INTERNATIONAL,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Both,NKRA,10%,M1,< 1%,W1,< 10
2,Castor oil,8001-79-4,8001794,CASRN,RPM INTERNATIONAL INC,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Commercial,No,100%,M4,60% – < 90%,W5,100 – < 500


In [12]:
log('Running initial audit checks.')

audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(dropna=True),
}).sort_values(['missing_count', 'unique_values'], ascending=[False, False])

log(f'Duplicate row count: {df.duplicated().sum():,}')
display(audit.head(25))
record_change('Completed first-pass audit of dtypes, missing values, and duplicates.')

[2026-06-20 05:57:59] Running initial audit checks.
[2026-06-20 05:58:00] Duplicate row count: 1,374


,dtype,missing_count,missing_pct,unique_values
JOINT FUNCT CAT OTHER DESC,float64,64023,100.00,0
CONS / COMM FUNCT CAT OTHER DESC,str,61451,95.98,824
CONS / COMM PROD CAT OTHER DESC,str,60998,95.28,1445
JOINT FC CODE,str,56640,88.47,39
JOINT FUNCTION CATEGORY,str,56640,88.47,39
FOREIGN PC STATE,str,52743,82.38,5
FOREIGN PC COUNTY / PARISH,str,51089,79.80,57
FOREIGN PC ADDRESS LINE2,str,50836,79.40,99
PF PPV – NKRA,str,50366,78.67,6
SITE NAICS ACTIVITY 3,str,50324,78.60,4


2026-06-20 05:58:00 | Completed first-pass audit of dtypes, missing values, and duplicates.


## Exercise 1: Standardize Column Names

Review the original headers first, then normalize them only if you are ready to work with cleaner names.

The goal is to make the notebook repeatable and easier to debug, not to silently change meaning.

In [13]:
def standardize_columns(columns: pd.Index) -> pd.Index:
    cleaned = (
        pd.Index(columns)
        .astype('string')
        .str.normalize('NFKC')
        .str.strip()
        .str.lower()
        .str.replace(r'[^0-9a-z]+', '_', regex=True)
        .str.replace(r'_+', '_', regex=True)
        .str.strip('_')
    )
    return cleaned


column_preview = pd.DataFrame({'original_column': df.columns[:25]})
display(column_preview)

df.columns = standardize_columns(df.columns)
log('Column names standardized.')
record_change('Standardized column names with a lower_snake_case pattern.')
display(pd.DataFrame({'cleaned_column': df.columns[:25]}))

,original_column
0,CHEMICAL NAME
1,CHEMICAL ID
2,CHEMICAL ID W/O DASHES
3,CHEMICAL ID TYPE
4,STANDARDIZED PARENT COMPANY NAME
5,FOREIGN PARENT COMPANY NAME
6,FOREIGN PC ADDRESS LINE1
7,FOREIGN PC ADDRESS LINE2
8,FOREIGN PC CITY
9,FOREIGN PC COUNTY / PARISH


[2026-06-20 05:58:00] Column names standardized.
2026-06-20 05:58:00 | Standardized column names with a lower_snake_case pattern.


,cleaned_column
0,chemical_name
1,chemical_id
2,chemical_id_w_o_dashes
3,chemical_id_type
4,standardized_parent_company_name
5,foreign_parent_company_name
6,foreign_pc_address_line1
7,foreign_pc_address_line2
8,foreign_pc_city
9,foreign_pc_county_parish


## Exercise 2: Clean Text Fields

Use this section to inspect encoding artifacts, stray whitespace, and inconsistent casing.

Work on a copy of the dataframe and save the original source untouched.

In [14]:
def normalize_text_series(series: pd.Series) -> pd.Series:
    return (
        series.astype('string')
        .str.normalize('NFKC')
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )


text_columns = df.select_dtypes(include='object').columns.tolist()
log(f'Found {len(text_columns):,} object-type columns.')

sample_rows = []
for column in text_columns[:20]:
    non_null = df[column].dropna()
    sample_value = non_null.iloc[0] if not non_null.empty else ''
    sample_rows.append({'column': column, 'sample_value': sample_value})

display(pd.DataFrame(sample_rows))

text_artifact_count = 0
for column in text_columns:
    text_artifact_count += df[column].astype('string').str.contains('Ã', na=False).sum()

log(f'Potential mojibake hits containing "Ã": {int(text_artifact_count):,}')
record_change('Inspected text columns for whitespace and encoding artifacts.')

[2026-06-20 05:58:00] Found 83 object-type columns.


C:\Users\Willaim\AppData\Local\Temp\ipykernel_20080\1185533554.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include='object').columns.tolist()


,column,sample_value
0,chemical_name,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis..."
1,chemical_id,4193-55-9
2,chemical_id_type,CASRN
3,standardized_parent_company_name,TEH FONG MIN INTERNATIONAL
4,foreign_parent_company_name,SHELL PLC
5,foreign_pc_address_line1,Shell Centre
6,foreign_pc_address_line2,SE17 NA
7,foreign_pc_city,London
8,foreign_pc_county_parish,CBI
9,foreign_pc_state,CBI


[2026-06-20 05:58:01] Potential mojibake hits containing "Ã": 0
2026-06-20 05:58:01 | Inspected text columns for whitespace and encoding artifacts.


## Exercise 3: Build Ontology Seed Entity Lists

Extract stable chemical and company entities from the wide source sheet.

Keep the extraction explicit so you can inspect the exact columns that contributed to each entity list.

In [15]:
def normalize_entity_text(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    text = pd.Series([text], dtype='string').str.normalize('NFKC').iloc[0]
    text = str(text).replace('\u00a0', ' ')
    text = ' '.join(text.split())
    return text


def build_entity_list(frame: pd.DataFrame, entity_type: str, name_column: str, id_columns: list[str]) -> pd.DataFrame:
    log(f'Building {entity_type} entity list from {name_column} and {len(id_columns)} identifier columns.')
    available_id_columns = [column for column in id_columns if column in frame.columns]
    missing_id_columns = sorted(set(id_columns) - set(available_id_columns))
    if missing_id_columns:
        log(f'{entity_type}: skipped missing columns: {missing_id_columns}')

    selected_columns = [name_column] + available_id_columns
    subset = frame[selected_columns].copy()
    subset[name_column] = subset[name_column].map(normalize_entity_text)

    for column in available_id_columns:
        subset[column] = subset[column].map(normalize_entity_text)

    subset = subset[subset[name_column].notna()].copy()
    subset['entity_type'] = entity_type
    subset['canonical_entity_name'] = subset[name_column]
    if available_id_columns:
        subset['canonical_entity_key'] = subset[available_id_columns].bfill(axis=1).iloc[:, 0]
    else:
        subset['canonical_entity_key'] = subset[name_column]
    subset['canonical_entity_key'] = subset['canonical_entity_key'].fillna(subset[name_column])
    subset['source_columns'] = ', '.join(selected_columns)

    duplicate_count = int(subset.duplicated(subset=['canonical_entity_key', 'canonical_entity_name']).sum())
    log(f'{entity_type}: extracted {len(subset):,} rows before deduplication; duplicate key/name pairs: {duplicate_count:,}')

    entity_frame = (
        subset[[
            'entity_type',
            'canonical_entity_key',
            'canonical_entity_name',
            'source_columns',
            *available_id_columns,
        ]]
        .drop_duplicates(subset=['canonical_entity_key', 'canonical_entity_name'])
        .sort_values(['canonical_entity_name', 'canonical_entity_key'], na_position='last')
        .reset_index(drop=True)
    )

    log(f'{entity_type}: finalized {len(entity_frame):,} unique entities.')
    display(entity_frame.head(10))
    return entity_frame


chemical_entities = build_entity_list(
    df,
    entity_type='chemical',
    name_column='chemical_name',
    id_columns=['chemical_id', 'chemical_id_w_o_dashes', 'chemical_id_type'],
)

company_entities = build_entity_list(
    df,
    entity_type='company',
    name_column='standardized_parent_company_name',
    id_columns=['foreign_parent_company_name', 'domestic_parent_company_name', 'foreign_pc_dun_bradstreet_number', 'domestic_pc_dun_bradstreet_number'],
)

ontology_entities = pd.concat([chemical_entities, company_entities], ignore_index=True)
log(f'Ontology seed size: {len(ontology_entities):,} rows total.')
record_change('Built chemical and company ontology seed lists from the source sheet.')
entity_counts = ontology_entities['entity_type'].value_counts().reset_index()
entity_counts.columns = ['entity_type', 'entity_count']
display(entity_counts)
display(ontology_entities.head(20))

[2026-06-20 05:58:01] Building chemical entity list from chemical_name and 3 identifier columns.
[2026-06-20 05:58:27] chemical: extracted 64,023 rows before deduplication; duplicate key/name pairs: 55,406
[2026-06-20 05:58:27] chemical: finalized 8,617 unique entities.


,entity_type,canonical_entity_key,canonical_entity_name,source_columns,chemical_id,chemical_id_w_o_dashes,chemical_id_type
0,chemical,24103,"(Polyisobutenyl)dihydro-2,5-furandione reactio...","chemical_name, chemical_id, chemical_id_w_o_da...",24103,24103,Accession Number
1,chemical,18805,"(Polyisobutenyl)dihydro-2,5-furandione, reacti...","chemical_name, chemical_id, chemical_id_w_o_da...",18805,18805,Accession Number
2,chemical,9000-90-2,.alpha.-Amylase,"chemical_name, chemical_id, chemical_id_w_o_da...",9000-90-2,9000902,CASRN
3,chemical,10016-20-3,.alpha.-Cyclodextrin,"chemical_name, chemical_id, chemical_id_w_o_da...",10016-20-3,10016203,CASRN
4,chemical,128446-33-3,".alpha.-Cyclodextrin, 2-hydroxypropyl ethers","chemical_name, chemical_id, chemical_id_w_o_da...",128446-33-3,128446333,CASRN
5,chemical,56038-13-2,".alpha.-D-Galactopyranoside, 1,6-dichloro-1,6-...","chemical_name, chemical_id, chemical_id_w_o_da...",56038-13-2,56038132,CASRN
6,chemical,57-50-1,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",57-50-1,57501,CASRN
7,chemical,12738-64-6,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",12738-64-6,12738646,CASRN
8,chemical,27216-37-1,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",27216-37-1,27216371,CASRN
9,chemical,37318-31-3,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",37318-31-3,37318313,CASRN


[2026-06-20 05:58:27] Building company entity list from standardized_parent_company_name and 4 identifier columns.
[2026-06-20 05:58:52] company: extracted 64,023 rows before deduplication; duplicate key/name pairs: 61,923
[2026-06-20 05:58:52] company: finalized 2,100 unique entities.


,entity_type,canonical_entity_key,canonical_entity_name,source_columns,foreign_parent_company_name,domestic_parent_company_name,foreign_pc_dun_bradstreet_number,domestic_pc_dun_bradstreet_number
0,company,"3D Systems, Inc.",3D SYSTEMS INC,"standardized_parent_company_name, foreign_pare...",NaN,"3D Systems, Inc.",NaN,17-357-4161
1,company,3M COMPANY,3M CO,"standardized_parent_company_name, foreign_pare...",NaN,3M COMPANY,NaN,00-617-3082
2,company,3V Sigma USA Inc.,3V SIGMA USA INC,"standardized_parent_company_name, foreign_pare...",NaN,3V Sigma USA Inc.,NaN,00-910-4381
3,company,"5N Plus, Inc",5N PLUS INC,"standardized_parent_company_name, foreign_pare...",NaN,"5N Plus, Inc",NaN,05-886-6989
4,company,CONCAST,A CUBED CORP,"standardized_parent_company_name, foreign_pare...",NaN,CONCAST,NaN,00-431-8853
5,company,CONCAST METAL PRODUCTS CO,A CUBED CORP,"standardized_parent_company_name, foreign_pare...",NaN,CONCAST METAL PRODUCTS CO,NaN,96-586-5046
6,company,"A.G. LAYNE, INC.",A G LAYNE INC,"standardized_parent_company_name, foreign_pare...",NaN,"A.G. LAYNE, INC.",NaN,02-854-1696
7,company,AO SMITH PCD,A O SMITH CORP,"standardized_parent_company_name, foreign_pare...",NaN,AO SMITH PCD,NaN,00-403-6208
8,company,A-Gas,A-GAS US HOLDINGS INC,"standardized_parent_company_name, foreign_pare...",NaN,A-Gas,NaN,08-113-2210
9,company,AAKASH CHEMICALS,AAKASH CHEMICALS,"standardized_parent_company_name, foreign_pare...",NaN,AAKASH CHEMICALS,NaN,01-687-5627


[2026-06-20 05:58:52] Ontology seed size: 10,717 rows total.
2026-06-20 05:58:52 | Built chemical and company ontology seed lists from the source sheet.


,entity_type,entity_count
0,chemical,8617
1,company,2100


,entity_type,canonical_entity_key,canonical_entity_name,source_columns,chemical_id,chemical_id_w_o_dashes,chemical_id_type,foreign_parent_company_name,domestic_parent_company_name,foreign_pc_dun_bradstreet_number,domestic_pc_dun_bradstreet_number
0,chemical,24103,"(Polyisobutenyl)dihydro-2,5-furandione reactio...","chemical_name, chemical_id, chemical_id_w_o_da...",24103,24103,Accession Number,NaN,NaN,NaN,NaN
1,chemical,18805,"(Polyisobutenyl)dihydro-2,5-furandione, reacti...","chemical_name, chemical_id, chemical_id_w_o_da...",18805,18805,Accession Number,NaN,NaN,NaN,NaN
2,chemical,9000-90-2,.alpha.-Amylase,"chemical_name, chemical_id, chemical_id_w_o_da...",9000-90-2,9000902,CASRN,NaN,NaN,NaN,NaN
3,chemical,10016-20-3,.alpha.-Cyclodextrin,"chemical_name, chemical_id, chemical_id_w_o_da...",10016-20-3,10016203,CASRN,NaN,NaN,NaN,NaN
4,chemical,128446-33-3,".alpha.-Cyclodextrin, 2-hydroxypropyl ethers","chemical_name, chemical_id, chemical_id_w_o_da...",128446-33-3,128446333,CASRN,NaN,NaN,NaN,NaN
5,chemical,56038-13-2,".alpha.-D-Galactopyranoside, 1,6-dichloro-1,6-...","chemical_name, chemical_id, chemical_id_w_o_da...",56038-13-2,56038132,CASRN,NaN,NaN,NaN,NaN
6,chemical,57-50-1,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",57-50-1,57501,CASRN,NaN,NaN,NaN,NaN
7,chemical,12738-64-6,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",12738-64-6,12738646,CASRN,NaN,NaN,NaN,NaN
8,chemical,27216-37-1,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",27216-37-1,27216371,CASRN,NaN,NaN,NaN,NaN
9,chemical,37318-31-3,".alpha.-D-Glucopyranoside, .beta.-D-fructofura...","chemical_name, chemical_id, chemical_id_w_o_da...",37318-31-3,37318313,CASRN,NaN,NaN,NaN,NaN


## Exercise 4: Normalize Companies, Chemicals, and Facts

Normalize company names first, then chemical IDs, then flatten the workbook into traceable long-form tables.

Keep source row pointers on every derived fact so you can trace each record back to the original workbook row.

In [16]:
working_df = df.reset_index().rename(columns={'index': 'source_row_id'})
log(f'Created working dataframe with source_row_id values for {len(working_df):,} source rows.')

company_name_columns = [
    'standardized_parent_company_name',
    'foreign_parent_company_name',
    'domestic_parent_company_name',
]

for column in company_name_columns:
    if column in working_df.columns:
        normalized_column = f"{column.lower().replace(' ', '_').replace('/', '_').replace('&', 'and')}_normalized"
        working_df[normalized_column] = working_df[column].map(normalize_entity_text)
        log(f'Normalized company names into {normalized_column}.')

chemical_id_columns = [
    'chemical_id',
    'chemical_id_w_o_dashes',
    'chemical_id_type',
]

for column in chemical_id_columns:
    if column in working_df.columns:
        normalized_column = f"{column.lower().replace(' ', '_').replace('/', '_').replace('&', 'and')}_normalized"
        working_df[normalized_column] = working_df[column].map(normalize_entity_text)
        log(f'Normalized chemical identifiers into {normalized_column}.')

working_df['chemical_name_normalized'] = working_df['chemical_name'].map(normalize_entity_text)
log('Normalized chemical names into chemical_name_normalized.')

company_index = (
    working_df[[
        'source_row_id',
        'standardized_parent_company_name',
        'standardized_parent_company_name_normalized' if 'standardized_parent_company_name_normalized' in working_df.columns else 'standardized_parent_company_name',
        'foreign_parent_company_name',
        'domestic_parent_company_name',
        'foreign_pc_dun_bradstreet_number',
        'domestic_pc_dun_bradstreet_number',
    ]]
    .drop_duplicates(subset=['standardized_parent_company_name_normalized' if 'standardized_parent_company_name_normalized' in working_df.columns else 'standardized_parent_company_name'])
    .reset_index(drop=True)
)
log(f'Built company_index with {len(company_index):,} unique rows.')
display(company_index.head(10))

chemical_index = (
    working_df[[
        'source_row_id',
        'chemical_name',
        'chemical_name_normalized',
        'chemical_id',
        'chemical_id_w_o_dashes',
        'chemical_id_type',
        'chemical_id_normalized' if 'chemical_id_normalized' in working_df.columns else 'chemical_id',
        'chemical_id_w_o_dashes_normalized' if 'chemical_id_w_o_dashes_normalized' in working_df.columns else 'chemical_id_w_o_dashes',
    ]]
    .drop_duplicates(subset=['chemical_id_w_o_dashes_normalized' if 'chemical_id_w_o_dashes_normalized' in working_df.columns else 'chemical_id_w_o_dashes', 'chemical_name_normalized' if 'chemical_name_normalized' in working_df.columns else 'chemical_name'])
    .reset_index(drop=True)
)
log(f'Built chemical_index with {len(chemical_index):,} unique rows.')
display(chemical_index.head(10))

volume_columns = [
    '2023_domestic_pv',
    '2023_import_pv',
    '2023_pv',
    '2022_pv',
    '2021_pv',
    '2020_pv',
    '2023_v_used_on_site',
    '2023_v_exported',
    '2023_nationally_aggregated_pv',
    '2022_nationally_aggregated_pv',
    '2021_nationally_aggregated_pv',
    '2020_nationally_aggregated_pv',
    'c_c_pv_pct',
]
available_volume_columns = [column for column in volume_columns if column in working_df.columns]
log(f'Unpivoting {len(available_volume_columns):,} volume columns into long form.')
volume_long = working_df.melt(
    id_vars=[
        'source_row_id',
        'standardized_parent_company_name',
        'standardized_parent_company_name_normalized' if 'standardized_parent_company_name_normalized' in working_df.columns else 'standardized_parent_company_name',
        'chemical_name',
        'chemical_name_normalized',
        'chemical_id',
        'chemical_id_normalized',
        'chemical_id_w_o_dashes',
        'chemical_id_w_o_dashes_normalized',
        'chemical_id_type',
        'activity',
        'cons_and_or_comm_use',
        'used_in_prod_for_children',
        'pct_byp_code',
        'percent_byproduct',
        'workers_code',
        'workers_reasonably_likely_exposed',
        'max_conc_code',
        'maximum_concentration',
        'c_c_prod_cat_code',
        'consumer_commercial_product_category',
        'c_c_fc_code',
        'consumer_commercial_function_category',
        'joint_fc_code',
        'joint_function_category',
        'physical_form_s_list',
    ],
    value_vars=available_volume_columns,
    var_name='quantity_column',
    value_name='quantity_value_raw',
).dropna(subset=['quantity_value_raw'])
log(f'Created volume_long with {len(volume_long):,} populated quantity rows.')

physical_form_flag_columns = [column for column in working_df.columns if column.startswith('pf_ppv')]
physical_form_source = working_df[[
    'source_row_id',
    'chemical_name',
    'chemical_id',
    'physical_form_s_list',
    *physical_form_flag_columns,
]].copy()
physical_form_rows = []
for _, row in physical_form_source.iterrows():
    raw_forms = row['physical_form_s_list']
    forms = []
    if pd.notna(raw_forms):
        forms = [normalize_entity_text(part) for part in str(raw_forms).split(';')]
        forms = [form for form in forms if form]
    if not forms:
        forms = [column for column in physical_form_flag_columns if pd.notna(row.get(column))]
    for form in forms:
        physical_form_rows.append({
            'source_row_id': row['source_row_id'],
            'chemical_name': row['chemical_name'],
            'chemical_id': row['chemical_id'],
            'physical_form': form,
        })
physical_form_atomic = pd.DataFrame(physical_form_rows).drop_duplicates().reset_index(drop=True)
log(f'Created physical_form_atomic with {len(physical_form_atomic):,} atomic rows.')
display(physical_form_atomic.head(20))

company_chemical_fact = volume_long.copy()
company_chemical_fact['company_name'] = company_chemical_fact['standardized_parent_company_name_normalized']
company_chemical_fact['chemical_id_canonical'] = company_chemical_fact['chemical_id_w_o_dashes_normalized'].fillna(company_chemical_fact['chemical_id_normalized'])
log(f'Built company_chemical_fact with {len(company_chemical_fact):,} traceable fact rows.')
display(company_chemical_fact.head(20))
record_change('Normalized company and chemical identity fields, unpivoted volume columns, and split physical forms into atomic rows.')

[2026-06-20 05:58:52] Created working dataframe with source_row_id values for 64,023 source rows.
[2026-06-20 05:58:58] Normalized company names into standardized_parent_company_name_normalized.
[2026-06-20 05:59:00] Normalized company names into foreign_parent_company_name_normalized.
[2026-06-20 05:59:05] Normalized company names into domestic_parent_company_name_normalized.
[2026-06-20 05:59:11] Normalized chemical identifiers into chemical_id_normalized.
[2026-06-20 05:59:16] Normalized chemical identifiers into chemical_id_w_o_dashes_normalized.
[2026-06-20 05:59:21] Normalized chemical identifiers into chemical_id_type_normalized.
[2026-06-20 05:59:27] Normalized chemical names into chemical_name_normalized.
[2026-06-20 05:59:27] Built company_index with 1,444 unique rows.


,source_row_id,standardized_parent_company_name,standardized_parent_company_name_normalized,foreign_parent_company_name,domestic_parent_company_name,foreign_pc_dun_bradstreet_number,domestic_pc_dun_bradstreet_number
0,0,TEH FONG MIN INTERNATIONAL,TEH FONG MIN INTERNATIONAL,NaN,"TFM North America, Inc.",NaN,60-264-5305
1,2,RPM INTERNATIONAL INC,RPM INTERNATIONAL INC,NaN,RPM International Inc.,NaN,00-321-8484
2,4,SHELL PLC,SHELL PLC,SHELL PLC,Shell Petroleum Inc.,42-379-2808,00-429-4740
3,14,YUSHIRO INC,YUSHIRO INC,YUSHIRO CHEMICAL INDUSTRY CO LTD,"YUSHIRO MFG. AMERICA, INC.",69-062-7898,17-736-3157
4,22,CBI,CBI,CBI,CBI,CBI,CBI
5,25,HASTINGS UTILITIES / PUBLIC POWER GENERATION A...,HASTINGS UTILITIES / PUBLIC POWER GENERATION A...,NaN,HASTINGS UTILITIES,NaN,13-515-9325
6,27,SK TILLEY HOLDING LP,SK TILLEY HOLDING LP,NaN,"Tilley Distribution, Inc.",NaN,96-707-7087
7,28,BERKSHIRE HATHAWAY INC,BERKSHIRE HATHAWAY INC,NaN,"Berkshire Hathaway, Inc.",NaN,00-102-4314
8,29,KANTO CHEMICAL CO TOKYO JAPAN,KANTO CHEMICAL CO TOKYO JAPAN,NaN,KANTO CORPORATION,NaN,86-866-8211
9,38,ENTERGY CORP,ENTERGY CORP,NaN,ENTERGY SERVICES INC.,NaN,80-974-9005


[2026-06-20 05:59:27] Built chemical_index with 8,557 unique rows.


,source_row_id,chemical_name,chemical_name_normalized,chemical_id,chemical_id_w_o_dashes,chemical_id_type,chemical_id_normalized,chemical_id_w_o_dashes_normalized
0,0,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...","Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,4193559,CASRN,4193-55-9,4193559
1,1,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...","1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,41098560,CASRN,41098-56-0,41098560
2,2,Castor oil,Castor oil,8001-79-4,8001794,CASRN,8001-79-4,8001794
3,4,"Distillates (petroleum), light thermal cracked","Distillates (petroleum), light thermal cracked",64741-82-8,64741828,CASRN,64741-82-8,64741828
4,5,"Naphtha (petroleum), full-range alkylate, buta...","Naphtha (petroleum), full-range alkylate, buta...",68527-27-5,68527275,CASRN,68527-27-5,68527275
5,6,"Distillates (petroleum), light catalytic cracked","Distillates (petroleum), light catalytic cracked",64741-59-9,64741599,CASRN,64741-59-9,64741599
6,7,"Naphtha (petroleum), heavy catalytic reformed","Naphtha (petroleum), heavy catalytic reformed",64741-68-0,64741680,CASRN,64741-68-0,64741680
7,8,Sulfur,Sulfur,7704-34-9,7704349,CASRN,7704-34-9,7704349
8,9,"Propane, 2-methyl-","Propane, 2-methyl-",75-28-5,75285,CASRN,75-28-5,75285
9,10,"Tail gas (petroleum), catalytic reformed napht...","Tail gas (petroleum), catalytic reformed napht...",68478-27-3,68478273,CASRN,68478-27-3,68478273


[2026-06-20 05:59:27] Unpivoting 13 volume columns into long form.
[2026-06-20 05:59:27] Created volume_long with 625,150 populated quantity rows.
[2026-06-20 05:59:35] Created physical_form_atomic with 64,490 atomic rows.


,source_row_id,chemical_name,chemical_id,physical_form
0,0,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,Dry Powder
1,1,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,Dry Powder
2,2,Castor oil,8001-79-4,Liquid
3,3,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,Dry Powder
4,8,Sulfur,7704-34-9,Liquid
5,9,"Propane, 2-methyl-",75-28-5,Gas Vapor
6,11,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",16470-24-9,Dry Powder
7,12,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",16470-24-9,Dry Powder
8,13,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,Dry Powder
9,14,"1,3-Benzenedicarboxylic acid, sodium salt (1:2)",10027-33-5,pf_ppv_liquid


[2026-06-20 05:59:35] Built company_chemical_fact with 625,150 traceable fact rows.


,source_row_id,standardized_parent_company_name,standardized_parent_company_name_normalized,chemical_name,chemical_name_normalized,chemical_id,chemical_id_normalized,chemical_id_w_o_dashes,chemical_id_w_o_dashes_normalized,chemical_id_type,...,consumer_commercial_product_category,c_c_fc_code,consumer_commercial_function_category,joint_fc_code,joint_function_category,physical_form_s_list,quantity_column,quantity_value_raw,company_name,chemical_id_canonical
8,8,SHELL PLC,SHELL PLC,Sulfur,Sulfur,7704-34-9,7704-34-9,7704349,7704349,CASRN,...,Vehicular or appliance fuels,F030,Fuel agents,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,7704349
9,9,SHELL PLC,SHELL PLC,"Propane, 2-methyl-","Propane, 2-methyl-",75-28-5,75-28-5,75285,75285,CASRN,...,Vehicular or appliance fuels,F030,Fuel agents,NaN,NaN,Gas Vapor,2023_domestic_pv,CBI,SHELL PLC,75285
14,14,YUSHIRO INC,YUSHIRO INC,"1,3-Benzenedicarboxylic acid, sodium salt (1:2)","1,3-Benzenedicarboxylic acid, sodium salt (1:2)",10027-33-5,10027-33-5,10027335,10027335,CASRN,...,NaN,NaN,NaN,NaN,NaN,NaN,2023_domestic_pv,"61,000",YUSHIRO INC,10027335
15,15,SHELL PLC,SHELL PLC,"Benzene, trimethyl-","Benzene, trimethyl-",25551-13-7,25551-13-7,25551137,25551137,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,25551137
16,16,SHELL PLC,SHELL PLC,"Benzene, (1-methylethyl)-","Benzene, (1-methylethyl)-",98-82-8,98-82-8,98828,98828,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,98828
17,17,SHELL PLC,SHELL PLC,"Benzene, ethyl-","Benzene, ethyl-",100-41-4,100-41-4,100414,100414,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,100414
18,18,SHELL PLC,SHELL PLC,"Benzene, ethenyl-","Benzene, ethenyl-",100-42-5,100-42-5,100425,100425,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,100425
19,19,SHELL PLC,SHELL PLC,Butane,Butane,106-97-8,106-97-8,106978,106978,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,106978
20,20,SHELL PLC,SHELL PLC,"1,3-Butadiene","1,3-Butadiene",106-99-0,106-99-0,106990,106990,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,106990
21,21,SHELL PLC,SHELL PLC,"1,2-Ethanediol","1,2-Ethanediol",107-21-1,107-21-1,107211,107211,CASRN,...,Vehicular or appliance fuels,NKRA,Not Known or Reasonably Ascertainable,NaN,NaN,Liquid,2023_domestic_pv,CBI,SHELL PLC,107211


2026-06-20 05:59:35 | Normalized company and chemical identity fields, unpivoted volume columns, and split physical forms into atomic rows.


## Exercise 5: Pre-Table Normalization Checklist

Use this checkpoint to confirm the workbook is ready for new tables before you split the model further.

The checklist should stay explicit and repeatable so it can be rerun after any source refresh.

In [17]:
checklist_rows = []

def add_check(name: str, status: str, details: str) -> None:
    checklist_rows.append({'check': name, 'status': status, 'details': details})


add_check(
    'Encoding artifacts',
    'needs_review' if any('Ã' in str(column) or 'â' in str(column) for column in working_df.columns) else 'ok',
    'Inspect any remaining mojibake in headers or range labels.',
)

add_check(
    'Company key',
    'ok' if 'standardized_parent_company_name_normalized' in working_df.columns else 'needs_review',
    'Use normalized parent company names as the primary company index key.',
)

add_check(
    'Chemical key',
    'ok' if 'chemical_id_w_o_dashes_normalized' in working_df.columns else 'needs_review',
    'Use a normalized chemical ID without dashes as the primary chemical key.',
)

code_label_pairs = [
    ('pct_byp_code', 'percent_byproduct'),
    ('workers_code', 'workers_reasonably_likely_exposed'),
    ('max_conc_code', 'maximum_concentration'),
    ('c_c_prod_cat_code', 'consumer_commercial_product_category'),
    ('c_c_fc_code', 'consumer_commercial_function_category'),
    ('joint_fc_code', 'joint_function_category'),
]
for code_column, label_column in code_label_pairs:
    if code_column in working_df.columns and label_column in working_df.columns:
        pair_status = 'ok' if working_df[[code_column, label_column]].drop_duplicates().shape[0] >= 1 else 'needs_review'
        add_check(
            f'{code_column} / {label_column}',
            pair_status,
            'Keep code and label columns together until lookup validation is complete.',
        )

sentinel_terms = ['CBI', 'NKRA']
for term in sentinel_terms:
    term_hits = int(working_df.astype('string').apply(lambda col: col.str.contains(term, na=False)).sum().sum())
    add_check(
        f'Sentinel {term}',
        'needs_review' if term_hits else 'ok',
        f'Found {term_hits:,} occurrences; confirm whether the term means confidential, unknown, or not applicable.',
    )

add_check(
    'Source row provenance',
    'ok' if 'source_row_id' in working_df.columns else 'needs_review',
    'Every derived table should keep source_row_id for traceability.',
)

add_check(
    'Atomic physical forms',
    'ok' if len(physical_form_atomic) >= 0 else 'needs_review',
    'Physical-form lists should be split into atomic rows before table creation.',
)

checklist_df = pd.DataFrame(checklist_rows)
log(f'Built normalization checklist with {len(checklist_df):,} checks.')
display(checklist_df)
record_change('Reviewed pre-table normalization checklist for encoding, keys, sentinels, provenance, and atomic field handling.')

[2026-06-20 05:59:37] Built normalization checklist with 13 checks.


,check,status,details
0,Encoding artifacts,ok,Inspect any remaining mojibake in headers or r...
1,Company key,ok,Use normalized parent company names as the pri...
2,Chemical key,ok,Use a normalized chemical ID without dashes as...
3,pct_byp_code / percent_byproduct,ok,Keep code and label columns together until loo...
4,workers_code / workers_reasonably_likely_exposed,ok,Keep code and label columns together until loo...
5,max_conc_code / maximum_concentration,ok,Keep code and label columns together until loo...
6,c_c_prod_cat_code / consumer_commercial_produc...,ok,Keep code and label columns together until loo...
7,c_c_fc_code / consumer_commercial_function_cat...,ok,Keep code and label columns together until loo...
8,joint_fc_code / joint_function_category,ok,Keep code and label columns together until loo...
9,Sentinel CBI,needs_review,"Found 767,087 occurrences; confirm whether the..."


2026-06-20 05:59:37 | Reviewed pre-table normalization checklist for encoding, keys, sentinels, provenance, and atomic field handling.


## Exercise 6: Start Canonical Tables

Use the normalized standardized parent company name as the company key and the undashed CAS number as the chemical key.

These first tables separate entity records from activity and quantity facts while keeping `source_row_id` for traceability.

In [18]:
company_table = (
    working_df[[
        'source_row_id',
        'standardized_parent_company_name',
        'standardized_parent_company_name_normalized',
        'foreign_parent_company_name',
        'foreign_parent_company_name_normalized',
        'domestic_parent_company_name',
        'domestic_parent_company_name_normalized',
        'foreign_pc_dun_bradstreet_number',
        'domestic_pc_dun_bradstreet_number',
    ]]
    .dropna(subset=['standardized_parent_company_name_normalized'])
    .drop_duplicates(subset=['standardized_parent_company_name_normalized'])
    .rename(columns={
        'standardized_parent_company_name': 'company_name_source',
        'standardized_parent_company_name_normalized': 'company_key',
    })
    .reset_index(drop=True)
)
log(f'Built company_table with {len(company_table):,} unique companies.')
display(company_table.head(10))

chemical_table = (
    working_df[[
        'source_row_id',
        'chemical_name',
        'chemical_name_normalized',
        'chemical_id',
        'chemical_id_normalized',
        'chemical_id_w_o_dashes',
        'chemical_id_w_o_dashes_normalized',
        'chemical_id_type',
        'chemical_id_type_normalized',
    ]]
    .dropna(subset=['chemical_id_w_o_dashes_normalized'])
    .drop_duplicates(subset=['chemical_id_w_o_dashes_normalized'])
    .rename(columns={
        'chemical_name': 'chemical_name_source',
        'chemical_name_normalized': 'chemical_name_key',
        'chemical_id': 'chemical_id_source',
        'chemical_id_normalized': 'chemical_id_key',
        'chemical_id_w_o_dashes': 'chemical_id_without_dashes_source',
        'chemical_id_w_o_dashes_normalized': 'chemical_cas_key',
        'chemical_id_type': 'chemical_id_type_source',
        'chemical_id_type_normalized': 'chemical_id_type_key',
    })
    .reset_index(drop=True)
)
log(f'Built chemical_table with {len(chemical_table):,} unique chemicals.')
display(chemical_table.head(10))

company_chemical_activity_fact = (
    working_df[[
        'source_row_id',
        'standardized_parent_company_name_normalized',
        'chemical_id_w_o_dashes_normalized',
        'chemical_name_normalized',
        'activity',
        'cons_and_or_comm_use',
        'used_in_prod_for_children',
        'pct_byp_code',
        'percent_byproduct',
        'workers_code',
        'workers_reasonably_likely_exposed',
        'max_conc_code',
        'maximum_concentration',
        'c_c_prod_cat_code',
        'consumer_commercial_product_category',
        'c_c_fc_code',
        'consumer_commercial_function_category',
        'joint_fc_code',
        'joint_function_category',
    ]]
    .drop_duplicates()
    .rename(columns={
        'standardized_parent_company_name_normalized': 'company_key',
        'chemical_id_w_o_dashes_normalized': 'chemical_cas_key',
        'chemical_name_normalized': 'chemical_name_key',
    })
    .reset_index(drop=True)
)
log(f'Built company_chemical_activity_fact with {len(company_chemical_activity_fact):,} rows.')
display(company_chemical_activity_fact.head(20))

quantity_columns = [
    '2023_domestic_pv',
    '2023_import_pv',
    '2023_pv',
    '2022_pv',
    '2021_pv',
    '2020_pv',
    '2023_v_used_on_site',
    '2023_v_exported',
    '2023_nationally_aggregated_pv',
    '2022_nationally_aggregated_pv',
    '2021_nationally_aggregated_pv',
    '2020_nationally_aggregated_pv',
    'c_c_pv_pct',
]
quantity_columns = [column for column in quantity_columns if column in working_df.columns]
quantity_fact = working_df.melt(
    id_vars=['source_row_id', 'standardized_parent_company_name_normalized', 'chemical_id_w_o_dashes_normalized', 'chemical_name_normalized'],
    value_vars=quantity_columns,
    var_name='quantity_type',
    value_name='quantity_value_raw',
).dropna(subset=['quantity_value_raw'])
quantity_fact = quantity_fact.rename(columns={
    'quantity_value_raw': 'quantity_value',
    'standardized_parent_company_name_normalized': 'company_key',
    'chemical_id_w_o_dashes_normalized': 'chemical_cas_key',
    'chemical_name_normalized': 'chemical_name_key',
}).reset_index(drop=True)
quantity_fact['quantity_value_numeric'] = pd.to_numeric(quantity_fact['quantity_value'], errors='coerce')
log(f'Built quantity_fact with {len(quantity_fact):,} long-form rows.')
display(quantity_fact.head(20))

physical_form_fact = physical_form_atomic.rename(columns={
    'chemical_id': 'chemical_cas_key',
    'chemical_name': 'chemical_name_key',
}).reset_index(drop=True)
log(f'Built physical_form_fact with {len(physical_form_fact):,} atomic rows.')
display(physical_form_fact.head(20))

record_change('Started canonical company, chemical, activity, quantity, and physical-form tables using normalized keys.')

[2026-06-20 05:59:37] Built company_table with 1,444 unique companies.


,source_row_id,company_name_source,company_key,foreign_parent_company_name,foreign_parent_company_name_normalized,domestic_parent_company_name,domestic_parent_company_name_normalized,foreign_pc_dun_bradstreet_number,domestic_pc_dun_bradstreet_number
0,0,TEH FONG MIN INTERNATIONAL,TEH FONG MIN INTERNATIONAL,NaN,NaN,"TFM North America, Inc.","TFM North America, Inc.",NaN,60-264-5305
1,2,RPM INTERNATIONAL INC,RPM INTERNATIONAL INC,NaN,NaN,RPM International Inc.,RPM International Inc.,NaN,00-321-8484
2,4,SHELL PLC,SHELL PLC,SHELL PLC,SHELL PLC,Shell Petroleum Inc.,Shell Petroleum Inc.,42-379-2808,00-429-4740
3,14,YUSHIRO INC,YUSHIRO INC,YUSHIRO CHEMICAL INDUSTRY CO LTD,YUSHIRO CHEMICAL INDUSTRY CO LTD,"YUSHIRO MFG. AMERICA, INC.","YUSHIRO MFG. AMERICA, INC.",69-062-7898,17-736-3157
4,22,CBI,CBI,CBI,CBI,CBI,CBI,CBI,CBI
5,25,HASTINGS UTILITIES / PUBLIC POWER GENERATION A...,HASTINGS UTILITIES / PUBLIC POWER GENERATION A...,NaN,NaN,HASTINGS UTILITIES,HASTINGS UTILITIES,NaN,13-515-9325
6,27,SK TILLEY HOLDING LP,SK TILLEY HOLDING LP,NaN,NaN,"Tilley Distribution, Inc.","Tilley Distribution, Inc.",NaN,96-707-7087
7,28,BERKSHIRE HATHAWAY INC,BERKSHIRE HATHAWAY INC,NaN,NaN,"Berkshire Hathaway, Inc.","Berkshire Hathaway, Inc.",NaN,00-102-4314
8,29,KANTO CHEMICAL CO TOKYO JAPAN,KANTO CHEMICAL CO TOKYO JAPAN,NaN,NaN,KANTO CORPORATION,KANTO CORPORATION,NaN,86-866-8211
9,38,ENTERGY CORP,ENTERGY CORP,NaN,NaN,ENTERGY SERVICES INC.,ENTERGY SERVICES INC.,NaN,80-974-9005


[2026-06-20 05:59:37] Built chemical_table with 8,553 unique chemicals.


,source_row_id,chemical_name_source,chemical_name_key,chemical_id_source,chemical_id_key,chemical_id_without_dashes_source,chemical_cas_key,chemical_id_type_source,chemical_id_type_key
0,0,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...","Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,4193-55-9,4193559,4193559,CASRN,CASRN
1,1,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...","1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,41098-56-0,41098560,41098560,CASRN,CASRN
2,2,Castor oil,Castor oil,8001-79-4,8001-79-4,8001794,8001794,CASRN,CASRN
3,4,"Distillates (petroleum), light thermal cracked","Distillates (petroleum), light thermal cracked",64741-82-8,64741-82-8,64741828,64741828,CASRN,CASRN
4,5,"Naphtha (petroleum), full-range alkylate, buta...","Naphtha (petroleum), full-range alkylate, buta...",68527-27-5,68527-27-5,68527275,68527275,CASRN,CASRN
5,6,"Distillates (petroleum), light catalytic cracked","Distillates (petroleum), light catalytic cracked",64741-59-9,64741-59-9,64741599,64741599,CASRN,CASRN
6,7,"Naphtha (petroleum), heavy catalytic reformed","Naphtha (petroleum), heavy catalytic reformed",64741-68-0,64741-68-0,64741680,64741680,CASRN,CASRN
7,8,Sulfur,Sulfur,7704-34-9,7704-34-9,7704349,7704349,CASRN,CASRN
8,9,"Propane, 2-methyl-","Propane, 2-methyl-",75-28-5,75-28-5,75285,75285,CASRN,CASRN
9,10,"Tail gas (petroleum), catalytic reformed napht...","Tail gas (petroleum), catalytic reformed napht...",68478-27-3,68478-27-3,68478273,68478273,CASRN,CASRN


[2026-06-20 05:59:37] Built company_chemical_activity_fact with 64,023 rows.


...

[2026-06-20 05:59:38] Built quantity_fact with 625,150 long-form rows.


,source_row_id,company_key,chemical_cas_key,chemical_name_key,quantity_type,quantity_value,quantity_value_numeric
0,8,SHELL PLC,7704349,Sulfur,2023_domestic_pv,CBI,NaN
1,9,SHELL PLC,75285,"Propane, 2-methyl-",2023_domestic_pv,CBI,NaN
2,14,YUSHIRO INC,10027335,"1,3-Benzenedicarboxylic acid, sodium salt (1:2)",2023_domestic_pv,"61,000",NaN
3,15,SHELL PLC,25551137,"Benzene, trimethyl-",2023_domestic_pv,CBI,NaN
4,16,SHELL PLC,98828,"Benzene, (1-methylethyl)-",2023_domestic_pv,CBI,NaN
5,17,SHELL PLC,100414,"Benzene, ethyl-",2023_domestic_pv,CBI,NaN
6,18,SHELL PLC,100425,"Benzene, ethenyl-",2023_domestic_pv,CBI,NaN
7,19,SHELL PLC,106978,Butane,2023_domestic_pv,CBI,NaN
8,20,SHELL PLC,106990,"1,3-Butadiene",2023_domestic_pv,CBI,NaN
9,21,SHELL PLC,107211,"1,2-Ethanediol",2023_domestic_pv,CBI,NaN


[2026-06-20 05:59:38] Built physical_form_fact with 64,490 atomic rows.


,source_row_id,chemical_name_key,chemical_cas_key,physical_form
0,0,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,Dry Powder
1,1,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,Dry Powder
2,2,Castor oil,8001-79-4,Liquid
3,3,"1,4-Benzenedisulfonic acid, 2,2'-[1,2-ethenedi...",41098-56-0,Dry Powder
4,8,Sulfur,7704-34-9,Liquid
5,9,"Propane, 2-methyl-",75-28-5,Gas Vapor
6,11,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",16470-24-9,Dry Powder
7,12,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",16470-24-9,Dry Powder
8,13,"Benzenesulfonic acid, 2,2'-(1,2-ethenediyl)bis...",4193-55-9,Dry Powder
9,14,"1,3-Benzenedicarboxylic acid, sodium salt (1:2)",10027-33-5,pf_ppv_liquid


2026-06-20 05:59:38 | Started canonical company, chemical, activity, quantity, and physical-form tables using normalized keys.


## Exercise 7: Type Conversion and Validation

Replace the placeholder column names below with real columns after inspection.

The idea is to convert one field at a time and check the result immediately.

In [19]:
# Example placeholders. Replace these after you confirm the actual column names.
candidate_numeric_columns = []
candidate_text_columns = []

for column in candidate_numeric_columns:
    if column in df.columns:
        log(f'Converting {column} to numeric.')
        df[column] = pd.to_numeric(df[column], errors='coerce')

for column in candidate_text_columns:
    if column in df.columns:
        log(f'Normalizing text values in {column}.')
        df[column] = normalize_text_series(df[column])

display(df.dtypes.head(25))
record_change('Left placeholders for controlled type conversion and validation.')

chemical_name                          str
chemical_id                            str
chemical_id_w_o_dashes               int64
chemical_id_type                       str
standardized_parent_company_name       str
foreign_parent_company_name            str
foreign_pc_address_line1               str
foreign_pc_address_line2               str
foreign_pc_city                        str
foreign_pc_county_parish               str
foreign_pc_state                       str
foreign_pc_postal_code                 str
foreign_pc_country_code                str
foreign_pc_dun_bradstreet_number       str
domestic_parent_company_name           str
domestic_pc_address_line1              str
domestic_pc_address_line2              str
domestic_pc_city                       str
domestic_pc_county_parish              str
domestic_pc_state                      str
domestic_pc_postal_code                str
domestic_pc_dun_bradstreet_number      str
site_name                              str
site_addres

2026-06-20 05:59:38 | Left placeholders for controlled type conversion and validation.


## Exercise 8: Export a Working Copy

Keep exports versioned so you can compare versions later.

Start with a dry run, then flip `WRITE_OUTPUT` to `True` when you want to save.

In [20]:
WRITE_OUTPUT = False
output_path = OUTPUT_DIR / f"{DATA_PATH.stem}_working_{datetime.now():%Y%m%d_%H%M%S}.csv"

if WRITE_OUTPUT:
    log(f'Writing cleaned copy to {output_path}')
    df.to_csv(output_path, index=False)
    record_change(f'Exported working copy to {output_path.name}.')
else:
    log(f'Dry run only. Output would be written to: {output_path}')
    record_change('Ran export cell in dry-run mode.')

display(pd.DataFrame({'session_log': session_log}))

[2026-06-20 05:59:38] Dry run only. Output would be written to: c:\Users\Willaim\Desktop\src\Exce Workl Project\cleaned data\exports\2024 CDR Consumer and Commercial Use Information_clean_working_20260620_055938.csv
2026-06-20 05:59:38 | Ran export cell in dry-run mode.


,session_log
0,2026-06-20 05:57:58 | Notebook scaffold initia...
1,2026-06-20 05:57:59 | Loaded 2024 CDR Consumer...
2,2026-06-20 05:58:00 | Completed first-pass aud...
3,2026-06-20 05:58:00 | Standardized column name...
4,2026-06-20 05:58:01 | Inspected text columns f...
5,2026-06-20 05:58:52 | Built chemical and compa...
6,2026-06-20 05:59:35 | Normalized company and c...
7,2026-06-20 05:59:37 | Reviewed pre-table norma...
8,2026-06-20 05:59:38 | Started canonical compan...
9,2026-06-20 05:59:38 | Left placeholders for co...


## Session Log

Use this area to summarize what you changed during the session.

- Loaded source data
- Audited columns and missing values
- Standardized names
- Inspected text fields
- Prepared conversion and export steps